In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
RUN_SMP = True
RUN_MUSICGEN = False
RUN_AUDIOLDM2 = False
RUN_MGELDM = False
RUN_VAMPNET = False

# Base Paths
if RUN_SMP:
  BASE_INPUT_DIR = "/content/drive/MyDrive/Plagiarism-Detection-System/data/segment_smp"
else:
  BASE_INPUT_DIR = "/content/drive/MyDrive/Plagiarism-Detection-System/data/generated_audio"

BASE_OUTPUT_DIR = "/content/drive/MyDrive/Plagiarism-Detection-System/data/dsp_variants"

In [ ]:
import librosa
import soundfile as sf
import os
import glob
import time

def check_variants_exist(base_name, output_dir):
    """
    Checks if all 10 variants exist for specific .wav file.
    """
    expected_suffixes = [
        "_pitchD4.wav", "_pitchD2.wav", "_pitchU2.wav", "_pitchU4.wav",
        "_tempo090.wav", "_tempo095.wav", "_tempo105.wav", "_tempo110.wav",
        "_pitchU4_tempo110.wav", "_pitchD4_tempo090.wav"
    ]

    for suffix in expected_suffixes:
        expected_path = os.path.join(output_dir, f"{base_name}{suffix}")
        if os.path.exists(expected_path):
            return True

    return False

def create_variants(input_wav_path, output_dir, base_name):
    """
    Generates 10 controlled DSP modifications (8 Single + 2 Combined).
    """
    y, sr = librosa.load(input_wav_path, sr=None)

    pitch_mods = [(-4, "D4"), (-2, "D2"), (2, "U2"), (4, "U4")]
    tempo_mods = [(0.90, "090"), (0.95, "095"), (1.05, "105"), (1.10, "110")]

    # SINGLE MODIFICATIONS
    for n_steps, p_label in pitch_mods:
        y_pitch = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=n_steps)
        out_name = f"{base_name}_pitch{p_label}.wav"
        sf.write(os.path.join(output_dir, out_name), y_pitch, sr)

    for rate, t_label in tempo_mods:
        y_tempo = librosa.effects.time_stretch(y=y, rate=rate)
        out_name = f"{base_name}_tempo{t_label}.wav"
        sf.write(os.path.join(output_dir, out_name), y_tempo, sr)

    # COMBINED MODIFICATIONS
    y_ext_up_pitch = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=4)
    y_ext_up_final = librosa.effects.time_stretch(y=y_ext_up_pitch, rate=1.10)
    out_name_up = f"{base_name}_pitchU4_tempo110.wav"
    sf.write(os.path.join(output_dir, out_name_up), y_ext_up_final, sr)

    y_ext_down_pitch = librosa.effects.pitch_shift(y=y, sr=sr, n_steps=-4)
    y_ext_down_final = librosa.effects.time_stretch(y=y_ext_down_pitch, rate=0.90)
    out_name_down = f"{base_name}_pitchD4_tempo090.wav"
    sf.write(os.path.join(output_dir, out_name_down), y_ext_down_final, sr)

In [ ]:
models_to_run = []
if RUN_SMP: models_to_run.append("audio")
if RUN_MUSICGEN: models_to_run.append("musicgen")
if RUN_AUDIOLDM2: models_to_run.append("audioldm2")
if RUN_MGELDM: models_to_run.append("mgeldm")
if RUN_VAMPNET: models_to_run.append("vampnet")

for model_name in models_to_run:
    print(f"Starting DSP Augmentation for: {model_name.upper()}")

    # Output folder
    current_input_dir = os.path.join(BASE_INPUT_DIR, model_name)
    current_output_dir = os.path.join(BASE_OUTPUT_DIR, model_name)
    os.makedirs(current_output_dir, exist_ok=True)

    # Find all .wav files
    ai_files = glob.glob(os.path.join(current_input_dir, "*.wav"))
    print(f"Found {len(ai_files)} files in {current_input_dir}")

    for file_path in ai_files:
        base_name = os.path.splitext(os.path.basename(file_path))[0]

        # Check existence
        if check_variants_exist(base_name, current_output_dir):
            print(f"{base_name} - The file is already processed. Ignoring...")
            continue

        # Generation
        print(f"Processing: {base_name}")
        start_time = time.time()

        create_variants(file_path, current_output_dir, base_name)

        end_time = time.time()
        print(f"Time taken: {end_time - start_time:.2f} seconds")

print("\nDSP Augmentation Pipeline Completed!")

Starting DSP Augmentation for: AUDIO
Found 1219 files in /content/drive/MyDrive/Plagiarism-Detection-System/data/segment_smp/audio
pair_15_comp_75s - The file is already processed. Ignoring...
pair_15_comp_132s - The file is already processed. Ignoring...
pair_15_comp_142s - The file is already processed. Ignoring...
pair_15_comp_175s - The file is already processed. Ignoring...
pair_15_comp_185s - The file is already processed. Ignoring...
pair_15_ori_68s - The file is already processed. Ignoring...
pair_15_ori_144s - The file is already processed. Ignoring...
pair_15_ori_198s - The file is already processed. Ignoring...
pair_15_comp_84s - The file is already processed. Ignoring...
pair_15_comp_151s - The file is already processed. Ignoring...
pair_15_comp_195s - The file is already processed. Ignoring...
pair_15_ori_76s - The file is already processed. Ignoring...
pair_15_ori_152s - The file is already processed. Ignoring...
pair_15_ori_207s - The file is already processed. Ignoring.